## 1. Representation of a Text Document in Vector Space Model and Computing Similarity between two documents. 

In [1]:
from preprocess import TextProcessor


# Step 1: Define file paths
input_filepath1 = "C:/Users/DELL/OneDrive/Documents/IRS LAB/doc1.txt"
input_filepath2 = "C:/Users/DELL/OneDrive/Documents/IRS LAB/doc2.txt"

# Step 2: Process Document 1
processor1 = TextProcessor(input_filepath1)
processor1.process_all()

print("\n===== Document 1 Processing =====")
print("\nFinal Processed Words (Doc1):")
print(processor1.final_processed)
print("\nLength of Final Processed Words (Doc1):", processor1.get_length(processor1.final_processed))


# Step 3: Process Document 2
processor2 = TextProcessor(input_filepath2)
processor2.process_all()

print("\n\n===== Document 2 Processing =====")
print("\nFinal Processed Words (Doc2):")
print(processor2.final_processed)
print("\nLength of Final Processed Words (Doc2):", processor2.get_length(processor2.final_processed))


===== Document 1 Processing =====

Final Processed Words (Doc1):
['information', 'retrieve', 'ir', 'system', 'software', 'tool', 'design', 'help', 'user', 'find', 'relevant', 'information', 'large', 'often', 'unstructur', 'collection', 'datum', 'text', 'document', 'website', 'multimedia', 'content', 'unlike', 'tradition', 'database', 'work', 'structur', 'datum', 'u', 'exact', 'query', 'ir', 'system', 'focu', 'retriev', 'information', 'base', 'content’', 'relevance', 'user', 'queer', 'common', 'example', 'ir', 'system', 'search', 'engine', 'like', 'google', 'scan', 'billion', 'webe', 'page', 'return', 'useful', 'result', 'base', 'user', 'input', 'system', 'consist', 'key', 'component', 'document', 'collection', 'ie', 'dataset', 'index', 'module', 'ie', 'processe', 'organize', 'datum', 'efficient', 'search', 'oper', 'queer', 'processor', 'ie', 'interpret', 'user', 'query', 'rank', 'algorithm', 'ie', 'determine', 'relevance', 'document', 'queer', 'fin', 'result', 'display', 'user', 'inte

In [3]:
# Step 4: Vocabulary Builder
def build_vocabulary(doc1, doc2):
    vocab = []
    for word in doc1:
        if word not in vocab:
            vocab+=[word]
    for word in doc2:
        if word not in vocab:
            vocab+=[word]
    return vocab

final_doc1 = processor1.final_processed
final_doc2 = processor2.final_processed

vocabulary = build_vocabulary(final_doc1, final_doc2)
# print("\nVocabulary:\n", vocabulary)

In [5]:
# Step 5.1: Binary Vectorization
def binary_vectorize(doc, vocabulary):
    vector = []
    for vocab_word in vocabulary:
        if vocab_word in doc:
            vector += [1]
        else:
            vector += [0]
    return vector

vector1 = binary_vectorize(final_doc1, vocabulary)
vector2 = binary_vectorize(final_doc2, vocabulary)

# print("\nBinary Vector for Document 1:\n", vector1)
# print("\nBinary Vector for Document 2:\n", vector2)


# Step 5.2: Term Frequency Vectorization
def term_frequency_vectorize(doc, vocabulary):
    vector = []
    for vocab_word in vocabulary:
        count = 0
        for word in doc:
            if word == vocab_word:
                count += 1   # manual count
        vector += [count]
    return vector

tf_vector1 = term_frequency_vectorize(final_doc1, vocabulary)
tf_vector2 = term_frequency_vectorize(final_doc2, vocabulary)

# print("\nTF Vector for Document 1:\n", tf_vector1)
# print("\nTF Vector for Document 2:\n", tf_vector2)


# Step 5.3: IDF + TF-IDF Vectorization
def manual_log(x, terms=50):
    if x <= 0:
        return 0  # log not defined for <= 0
    # Transform into form 1+y for convergence
    y = (x - 1) / (x + 1)
    result = 0
    for n in range(terms):
        term = (1 / (2*n + 1)) * (y ** (2*n + 1))
        result += term
    return 2 * result   # ln(x)

# Compute IDF manually
def compute_idf_manual(vocabulary, docs):
    N = len(docs)  # total number of documents
    idf_values = []
    for vocab_word in vocabulary:
        doc_count = 0
        for doc in docs:
            if vocab_word in doc:
                doc_count += 1
        if doc_count == 0:
            idf = 0
        else:
            # idf = manual_log(N / doc_count)  # manual log
            idf = manual_log((N) / (1 + doc_count)) + 1
        idf_values+=[idf]
    return idf_values

# TF-IDF Vectorization
def tf_idf_vectorize(tf_vector, idf_values):
    tfidf_vector = []
    for i in range(len(tf_vector)):
        tfidf_vector.append(tf_vector[i] * idf_values[i])
    return tfidf_vector

idf_values = compute_idf_manual(vocabulary, [final_doc1, final_doc2])
# print("\nManual IDF Values:\n", idf_values)

tfidf_vector1 = tf_idf_vectorize(tf_vector1, idf_values)
tfidf_vector2 = tf_idf_vectorize(tf_vector2, idf_values)

# print("\nTF-IDF Vector for Document 1:\n", tfidf_vector1)
# print("\nTF-IDF Vector for Document 2:\n", tfidf_vector2)

In [7]:
# Step 6: Cosine Similarity
def dot_product(v1, v2):
    total = 0
    for i in range(len(v1)):
        total += v1[i] * v2[i]
    return total

def magnitude(v):
    total = 0
    for i in range(len(v)):
        total += v[i] * v[i]
    return total ** 0.5

def cosine_similarity_manual(v1, v2):
    dot = dot_product(v1, v2)
    mag1 = magnitude(v1)
    mag2 = magnitude(v2)
    if mag1 == 0 or mag2 == 0:
        return 0
    return dot / (mag1 * mag2)

bin_similarity = cosine_similarity_manual(vector1, vector2)
tf_similarity = cosine_similarity_manual(tf_vector1, tf_vector2)
tfidf_similarity = cosine_similarity_manual(tfidf_vector1, tfidf_vector2)

print("Binary: Cosine Similarity between doc1 and doc2:", bin_similarity)
print("TF: Cosine Similarity between doc1 and doc2:", tf_similarity)
print("TF-IDF: Cosine Similarity between doc1 and doc2:", tfidf_similarity)

Binary: Cosine Similarity between doc1 and doc2: 0.5204014838815378
TF: Cosine Similarity between doc1 and doc2: 0.6470510546212859
TF-IDF: Cosine Similarity between doc1 and doc2: 0.4470288312307028


In [9]:
# Store results in a dictionary (method → similarity)
similarities = {
    "Binary": bin_similarity,
    "TF": tf_similarity,
    "TF-IDF": tfidf_similarity
}

# Define your own max function for dictionary
def get_max_method(sim_dict: dict) -> tuple:
    best_method = None
    best_value = float("-inf")  # very small value initially
    
    for method, value in sim_dict.items():
        if value > best_value:
            best_value = value
            best_method = method
    
    return best_method, best_value


# Print all values
print("Method    Cosine Similarity")
for method, value in similarities.items():
    print(f"{method:8}  {value:.6f}")

# Find the best method using custom function
best_method, best_value = get_max_method(similarities)

print("\nThe best method is:", best_method, "with similarity =", f"{best_value:.6f}")


Method    Cosine Similarity
Binary    0.520401
TF        0.647051
TF-IDF    0.447029

The best method is: TF with similarity = 0.647051
